# Exercise 2: ImageFolder classifier with a learning-rate schedule

Closest to the live-coding format: JPG folders loaded with `torchvision.datasets.ImageFolder`, a CNN with global average pooling, AdamW with a step learning-rate schedule, and a per-class report.

**Interviewer:** "This notebook doesn't work, and once it runs the results look strange. Walk me through it, tell me what you're checking, and fix it."

There are **9 problems**: 3 crash the code, 6 are silent (the code runs, but the result is wrong).

**How to work** (same as the real interview)
1. Read the whole notebook first and say out loud what each part does.
2. Run the cells from the top and fix the crashes one at a time.
3. Once it runs, go through it again for the silent bugs: check shapes, dtypes, value ranges, split sizes, train vs eval mode, and whether the numbers make sense.

*Level: Interview level, about 40 minutes. In Colab: Runtime → Change runtime type → T4 GPU (it also runs on a CPU, just slower).*

In [ ]:
import os

import numpy as np
import torch
import torch.nn as nn
from PIL import Image
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

BATCH_SIZE = 64
EPOCHS = 4
LR = 3e-3

torch.manual_seed(0)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)


def find_jpg_dir():
    """Finds data/ct3_jpg near this notebook. If it is missing, builds it from OrganAMNIST (downloads on Colab)."""
    for d in ["data", "../data", "../../data", "/content/data"]:
        if os.path.isdir(os.path.join(d, "ct3_jpg", "train")):
            return os.path.join(d, "ct3_jpg")
    root = next((d for d in ["data", "../data", "../../data", "/content/data"]
                 if os.path.exists(os.path.join(d, "organamnist_64.npz"))), None)
    if root is None:
        import subprocess
        import sys
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "medmnist"], check=True)
        import medmnist
        root = "data"
        os.makedirs(root, exist_ok=True)
        medmnist.OrganAMNIST(split="train", download=True, size=64, root=root)
    npz = np.load(os.path.join(root, "organamnist_64.npz"))
    for split in ["train", "val", "test"]:
        imgs = npz[f"{split}_images"]                 # read each array once; every npz[...] access reloads it
        labels = npz[f"{split}_labels"].reshape(-1)
        for old, name in {0: "bladder", 3: "heart", 6: "liver"}.items():
            folder = os.path.join(root, "ct3_jpg", split, name)
            os.makedirs(folder, exist_ok=True)
            for i in np.where(labels == old)[0]:
                Image.fromarray(imgs[i]).save(os.path.join(folder, f"{split}_{i:05d}.jpg"), quality=95)
    return os.path.join(root, "ct3_jpg")


DATA = find_jpg_dir()
print("data:", DATA)

### Transforms

In [ ]:
train_tf = transforms.Compose([
    transforms.Grayscale(num_output_channels=1),
    transforms.RandomAffine(degrees=10, translate=(0.05, 0.05)),
    transforms.Normalize([0.5], [0.25]),
    transforms.ToTensor(),
])
eval_tf = transforms.Compose([
    transforms.Grayscale(num_output_channels=1),
    transforms.ToTensor(),
    transforms.Normalize([0.5], [0.25]),
])

### Data

In [ ]:
train_ds = datasets.ImageFolder(os.path.join(DATA, "train"), transform=train_tf)
val_ds = datasets.ImageFolder(os.path.join(DATA, "train"), transform=eval_tf)
CLASS_NAMES = ["liver", "heart", "bladder"]
print("class -> index:", train_ds.class_to_idx)
print("train:", len(train_ds), "val:", len(val_ds))

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
val_loader = DataLoader(val_ds, batch_size=256, shuffle=False, num_workers=0, drop_last=True)

### Model

In [ ]:
def block(cin, cout):
    return nn.Sequential(nn.Conv2d(cin, cout, 3, padding=1), nn.BatchNorm2d(cout), nn.ReLU(), nn.MaxPool2d(2))


class Net(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        self.features = nn.Sequential(block(1, 16), block(16, 32), block(32, 64))
        self.pool = nn.AdaptiveAvgPool2d(1)            # (B, 64, 8, 8) -> (B, 64, 1, 1), works for any image size
        self.fc = nn.Linear(32, num_classes)

    def forward(self, x):
        x = self.pool(self.features(x)).flatten(1)
        return self.fc(x)


model = Net(len(CLASS_NAMES)).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=2, gamma=0.3)

### Training and validation

In [ ]:
for epoch in range(1, EPOCHS + 1):
    model.train()
    running, n = 0.0, 0
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad
        loss = criterion(model(images), labels)
        loss.backward()
        optimizer.step()
        scheduler.step()
        running += loss.item() * images.size(0)
        n += images.size(0)
    lr = scheduler.get_last_lr()[0]

    model.eval()
    correct, total = 0, 0
    with torch.no_grad:
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)
            preds = model(images).argmax()
            correct += (preds == labels).sum().item()
            total += labels.size(0)
    print(f"epoch {epoch}: lr {lr:.5f} train loss {running / n:.3f} | val acc {correct / total:.3f} on {total} images")

### Per-class report

In [ ]:
@torch.no_grad()
def predict(model, loader):
    model.eval()
    ys, ps = [], []
    for images, labels in loader:
        ps.append(model(images.to(device)).argmax(dim=1).cpu())
        ys.append(labels)
    return torch.cat(ys), torch.cat(ps)


y_true, y_pred = predict(model, val_loader)
k = len(CLASS_NAMES)
confusion = torch.zeros(k, k, dtype=torch.long)        # rows = true class, columns = predicted class
for t, p in zip(y_true, y_pred):
    confusion[t, p] += 1
print(confusion)
recall = confusion.diag().float() / confusion.sum(dim=1).clamp(min=1)
precision = confusion.diag().float() / confusion.sum(dim=0).clamp(min=1)
for name, r, p in zip(CLASS_NAMES, recall, precision):
    print(f"{name:8s} recall {r:.3f}  precision {p:.3f}")

### Your notes
Write each bug like this:
> Bug: cell … → why it's wrong → my fix → crash or silent?